In [2]:
import torch
from types import SimpleNamespace
from huggingface_hub import hf_hub_download
from src.models.autoregressive_module import AutoregressiveLitModule
from src.data.single_peptide_datamodule import SinglePeptideDataModule
from src.data.preprocessing.encodings import get_encodings
from src.evaluation.plots.plot_energies import plot_energies

N_SAMPLES, BATCH = 10_000, 1000
STD = 0.35  # precomputed_std, configs/data/transferable/up_to_8aa.yaml


In [2]:
# 1. Load Robin. __init__ only reads datamodule.hparams.num_dimensions, so a stub is enough.
ckpt = hf_hub_download("danyalrehman17/robin-transferable", "robin.ckpt")
model = AutoregressiveLitModule.load_from_checkpoint(
    ckpt, datamodule=SimpleNamespace(hparams=SimpleNamespace(num_dimensions=3)),
    map_location="cuda", weights_only=False,
).eval()


robin.ckpt:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

/data/ab2926/projects/AutoBG/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'net' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['net'])`.
Unexpected arguments: (), {'mean_free_prior': True, 'undiscretize_method': 'uniform_noise', 'noise_scale': 1.0}


In [3]:

# 2. Robin's per-residue atom order (alanine only)
ROBIN_ORDER = ["N", "H", "H1", "H2", "H3", "CA", "HA", "CB", "HB1", "HB2", "HB3", "C", "O", "OXT"]

def robin_index(top):
    idx = []
    for res in top.residues:
        by_name = {a.name: a.index for a in res.atoms}
        idx += [by_name[n] for n in ROBIN_ORDER if n in by_name]
    assert sorted(idx) == list(range(top.n_atoms))
    return torch.tensor(idx)


In [4]:
@torch.no_grad()
def sample(enc, n):
    n_atoms, out = len(enc["atom_type"]), []
    for _ in range(n // BATCH):
        e = {k: v[None].repeat(BATCH, 1).cuda() for k, v in enc.items()}
        tok = model.net.generate(batch_size=BATCH, num_atoms=n_atoms, encodings=e,
                                 temperature=model.hparams.temperature, use_cache=True)
        x = model.discretize.undiscretize_with_noise(tok).reshape(BATCH, n_atoms, 3)
        out.append(x.cpu())
    return torch.cat(out) * STD  # nm, Robin atom order


In [ ]:
import os
import openmm.app as app, openmm.unit as u
from src.models.utils import get_symmetry_change

for seq in ["AAA", "AAAAAA"]:
    dm = SinglePeptideDataModule(
        repo_id="transferable-samplers/sequential-boltzmann-generators-data",
        data_dir="data", sequence=seq, temperature=310, num_dimensions=3, num_atoms=0,
    )
    dm.prepare_data(); dm.setup()
    pot = dm.setup_potential()

    # 3. Encodings in Robin order -> sample -> map back to PDB order
    idx = robin_index(dm.topology)
    enc = {k: (v if k == "seq_len" else v[idx]) for k, v in get_encodings(dm.topology).items()}
    x_robin = sample(enc, N_SAMPLES)
    x_pdb = torch.empty_like(x_robin); x_pdb[:, idx] = x_robin
    
    # 4. MD reference: strided 10k subsample of the test split, as in the paper's eval
    md_ref = dm.data_test.data
    md_ref = md_ref[:: len(md_ref) // N_SAMPLES][:N_SAMPLES]

    # Mirror-image samples are real: Robin can produce D-amino-acid samples, which the
    # repo's eval flips back. Their energy is identical, so MD would happily simulate the
    # wrong enantiomer, and it can never cross back. Fix them before saving.
    flip = get_symmetry_change(md_ref, x_pdb, dm.topology)     # all in nm, PDB atom order
    x_pdb[flip] *= -1
    still_bad = get_symmetry_change(md_ref, x_pdb, dm.topology)
    x_keep = x_pdb[~still_bad]

    out = f"robin_samples/{seq}"
    os.makedirs(out, exist_ok=True)
    for i, x in enumerate(x_keep.numpy()):
        with open(f"{out}/{seq}_{i:05d}.pdb", "w") as f:
            app.PDBFile.writeFile(dm.pdb.topology, x * u.nanometer, f)



Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

: 

In [46]:
import os
import mdtraj as md
import openmm.app as app, openmm.unit as u
import glob

FRAME = 2
files = glob.glob("robin_md_trajectories/AAA/*/traj.dcd")
samples_at_frame = []
for file in files:
    pdb_file = file.replace("traj.dcd", "minimized.pdb")
    positions = md.load_dcd(file, top=pdb_file).xyz[FRAME] * u.nanometer  
    samples_at_frame.append(torch.tensor(positions._value))


dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and

IndexError: index 2 is out of bounds for axis 0 with size 0

In [55]:
md_pdb = torch.cat([x.unsqueeze(0) for x in samples_at_frame])

In [56]:
md_pdb.shape

torch.Size([282, 33, 3])

In [18]:
import glob

PEPTIDE = "AAA"
files = glob.glob(f"robin_samples/{PEPTIDE}/*.pdb")

pdb = []
for file in files:
    pdb.append(app.PDBFile(file).getPositions(asNumpy=True))
    

In [35]:
v = torch.cat([torch.tensor(x._value).unsqueeze(0) for x in pdb])

In [37]:
robin_pdb = v

In [61]:
E_robin.shape

torch.Size([9925])

In [40]:
dm = SinglePeptideDataModule(
        repo_id="transferable-samplers/sequential-boltzmann-generators-data",
        data_dir="data", sequence="AAA", temperature=310, num_dimensions=3, num_atoms=0,
    )
dm.prepare_data(); dm.setup()
pot = dm.setup_potential()

E_robin = pot.energy(robin_pdb).flatten()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [62]:
md_ref = dm.data_test.data
md_ref = md_ref[:: len(md_ref) // 9925][:9925]

In [68]:
md_ref.shape


E_md_ref = pot.energy(md_ref).flatten()

In [67]:
E_md_ref.shape

torch.Size([282])

In [57]:
E_md_sampled = pot.energy(md_pdb).flatten()

In [69]:
plot_energies(lambda fig, name: fig.savefig(f"AAA_{name}_true_robin_and_frames.png"), E_md_ref, E_robin, E_md_sampled, None, prefix="")

In [ ]:
E_md, E_robin = pot.energy(md_ref).flatten(), pot.energy(x_pdb).flatten()  # units of kT
plot_energies(lambda fig, name: fig.savefig(f"{seq}_{name}.png"), E_md, E_robin, None, None, prefix="")